# COSMOS Sérsic linear probe — jwst_dino teacher

Freeze the teacher backbone, extract embeddings over COSMOS f150w cutouts, and
linearly probe the **single-Sérsic fit labels** cross-matched from the photom
catalog by row id — Sérsic index `n` (→ log₁₀), effective radius `radius_sersic`
(→ log₁₀ px), and axis ratio `axratio_sersic`. Position angle is excluded
(circular target).

## The sample: one filter for all three labels

The catalog fits, not the encoder, set the ceiling here, so the sample is a
**single scalar threshold τ** shared by all three labels. Each catalog `*_err` is
put on the same scale — the fractional error in dex,

    σ_dex(x) = err(x) / (|x| · ln 10)      (τ = 0.02 dex ≈ 4.6% relative error)

and a source is kept when the **worst** of its three σ_dex is under τ:

    max( σ_dex(n), σ_dex(R_e), σ_dex(q) ) ≤ τ

plus two non-error cuts: `R_e ≥ MIN_REFF_PX` (resolved), and *railed* fits removed
(parameters pinned at the fitter bounds — `n` at 0.3 / 8.4, `q` at 0 / 1 — which
carry tiny formal errors, so no error threshold catches them). One τ ⇒ one sample
⇒ the three targets are probed on identical rows and their R² are comparable.
τ = 0.02 was chosen from a quality scan (R² keeps rising as τ tightens while the
target variances stay flat); lower τ trades sample size for cleaner labels.

## Readout & probe

Embedding is `concat(CLS, patch-mean)` (patch-mean alone underperforms CLS; the
concat adds ~0.01 R²). The probe is closed-form `RidgeCV` — no training steps, α
by internal CV — which measures whether the label is *linearly readable* from the
frozen features. Note the probe RMSE (~0.1 dex) is set by the image-information
limit, not by τ: the label formal error (~0.01 dex) is negligible in quadrature,
so a tighter τ cannot push RMSE toward τ.

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
import warnings; warnings.simplefilter('ignore')

sys.path.insert(0, os.path.join('..', '..'))          # jwst_dino/ (model + data import)
from model.jwst_dino import load_teacher_backbone
from data.augmentations import AsinhStretch

ROOT   = '~/ssl_outthere/data/image'
CKPT   = '/home/yacheng/ssl_outthere/encoder_image/jwst_dino/outputs/jwst_dino_ps6_st3/version_6/checkpoints/last.ckpt'
PHOTOM = '../../../../data/survey/cosmos_2025/COSMOSWeb_mastercatalog_v1_photom_primary.fits'

MIN_REFF_PX = 3.0       # resolved-source floor
TAU_DEX     = 0.02      # unified cut: max_j sigma_dex(label_j) <= TAU_DEX (~4.6% rel err)
MAX_SAMPLES = 30000     # cap the clean sample for speed (-1 = all)
TEST_FRAC   = 0.3
SEED        = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Regression targets (angle excluded): (photom column, plot label, log10 target?)
TARGETS = [('sersic',         r'$\log_{10} n$',         True),
           ('radius_sersic',  r'$\log_{10}\,R_e$ [px]', True),
           ('axratio_sersic', r'axis ratio $q$',        False)]

# Fitter bounds; a value pinned at a bound is a failed fit, not a measurement.
RAIL_BOUNDS = {'sersic': (0.31, 8.30), 'axratio_sersic': (0.05, 0.95)}
print('device:', DEVICE)

In [ ]:
# frozen teacher backbone (crop_size comes from the checkpoint's training config)
net  = load_teacher_backbone(CKPT, DEVICE)
crop = net.crop_size
print('crop_size:', crop)

In [ ]:
DEG_TO_PIX = 3600 * 1000 / 30.0   # radius_sersic[deg] -> px at 30 mas/pix
LN10 = np.log(10)


class CosmosSersicDataset(Dataset):
    """COSMOS f150w cutouts paired with clean single-Sérsic fit labels.

    id cross-match: image_index `id` = photom row index. Labels are stacked
    (N, 3) in `targets` order. A source is kept only if it is resolved, not
    railed, and its worst per-label sigma_dex = err/(|value|*ln10) is <= tau_dex.
    """

    def __init__(self, root, photom_catalog, targets, rail_bounds, filter='f150w',
                 crop_size=72, min_reff_px=3.0, tau_dex=0.02, max_samples=-1,
                 Q=20.0, scale=1.0, seed=42, verbose=True):
        self.root = os.path.expandvars(os.path.expanduser(root))
        self.center_crop = transforms.CenterCrop(crop_size)
        self.stretch = AsinhStretch(scale=scale, Q=Q, return_channel_pos=0)
        self.shards = {}

        index = Table.read(os.path.join(self.root, f'image_index_cosmos_{filter}.fits'))
        ids       = np.asarray(index['id'], np.int64)
        rel_path  = np.asarray(index['rel_path']).astype(str)
        local_idx = np.asarray(index['local_idx'], np.int64)

        ph = fits.open(os.path.expandvars(os.path.expanduser(photom_catalog)), memmap=True)[1].data
        get = lambda c: np.asarray(ph[c], np.float64)[ids]
        val = {'sersic': get('sersic'), 'radius_sersic': get('radius_sersic') * DEG_TO_PIX,
               'axratio_sersic': get('axratio_sersic')}
        err = {'sersic': get('sersic_err'), 'radius_sersic': get('radius_sersic_err') * DEG_TO_PIX,
               'axratio_sersic': get('axratio_sersic_err')}
        cols = [c for c, *_ in targets]

        resolved = (np.isfinite(val['radius_sersic']) & (val['radius_sersic'] >= min_reff_px)
                    & np.isfinite(val['sersic']) & np.isfinite(val['axratio_sersic'])
                    & (val['axratio_sersic'] > 0))
        railed = np.zeros(len(ids), bool)
        for c, (lo, hi) in rail_bounds.items():
            railed |= (val[c] <= lo) | (val[c] >= hi)

        sigma_dex = np.stack([err[c] / (np.abs(val[c]) * LN10) for c in cols], 1)
        usable = resolved & ~railed & np.isfinite(sigma_dex).all(1)
        clean = usable & (np.where(usable, sigma_dex.max(1), np.inf) <= tau_dex)

        keep = np.where(clean)[0]
        rng = np.random.default_rng(seed)
        if 0 < max_samples < len(keep):
            keep = rng.choice(keep, size=max_samples, replace=False)

        labels = np.stack([np.log10(val[c]) if log else val[c]
                           for c, _, log in targets], 1).astype(np.float32)
        self._samples = [(rel_path[i], local_idx[i]) for i in keep]
        self._labels  = labels[keep]
        if verbose:
            print(f'CosmosSersic [{filter}] — {len(ids)} cutouts, {resolved.sum()} resolved, '
                  f'{clean.sum()} clean (tau<={tau_dex}), {len(keep)} used')

    def _shard(self, rp):
        if rp not in self.shards:
            self.shards[rp] = np.load(os.path.join(self.root, rp), mmap_mode='r')
        return self.shards[rp]

    def __len__(self):
        return len(self._samples)

    def __getitem__(self, i):
        rp, li = self._samples[i]
        img = np.nan_to_num(self._shard(rp)[li].astype(np.float32))[None]   # (1, H, W)
        img = self.center_crop(torch.from_numpy(img)).numpy()
        return torch.from_numpy(self.stretch(img)), self._labels[i]


ds = CosmosSersicDataset(ROOT, PHOTOM, TARGETS, RAIL_BOUNDS, crop_size=crop,
                         min_reff_px=MIN_REFF_PX, tau_dex=TAU_DEX,
                         max_samples=MAX_SAMPLES, seed=SEED)
loader = DataLoader(ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

In [ ]:
# extract embeddings: concat(CLS, patch-mean)
@torch.no_grad()
def extract(net, loader, device):
    embs, labels = [], []
    for imgs, ys in tqdm(loader, desc='embeddings'):
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16,
                            enabled=device.type == 'cuda'):
            out = net(imgs.to(device))
        embs.append(torch.cat([out['cls'], out['patch'].mean(1)], 1).float().cpu().numpy())
        labels.append(np.asarray(ys))
    return np.concatenate(embs), np.concatenate(labels)

X, Y = extract(net, loader, DEVICE)
print('embeddings:', X.shape, ' labels:', Y.shape)

In [ ]:
# linear probe: standardize -> closed-form RidgeCV, one head per target
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

ALPHAS = np.logspace(-2, 5, 15)
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=TEST_FRAC, random_state=SEED)
scaler = StandardScaler().fit(Xtr)
Ztr, Zte = scaler.transform(Xtr), scaler.transform(Xte)

preds = np.zeros_like(Yte)
hdr = f'{"target":16s} {"std":>7s} {"R2":>7s} {"MAE":>7s} {"RMSE":>7s}'
print(f'=== Sérsic linear probe  (tau<={TAU_DEX} dex, {len(Xtr)} train / {len(Xte)} test) ===')
print(hdr); print('-' * len(hdr))
for j, (col, _, _) in enumerate(TARGETS):
    reg = RidgeCV(alphas=ALPHAS).fit(Ztr, Ytr[:, j])
    preds[:, j] = reg.predict(Zte)
    rmse = np.sqrt(np.mean((Yte[:, j] - preds[:, j]) ** 2))
    print(f'{col:16s} {Yte[:, j].std():7.3f} {r2_score(Yte[:, j], preds[:, j]):7.3f} '
          f'{mean_absolute_error(Yte[:, j], preds[:, j]):7.3f} {rmse:7.3f}')

In [ ]:
# pred-vs-true, NeurIPS poster style (matches LowResPT/neurips_spectrum_bench)
import matplotlib as mpl

POSTER_RC = {
    'font.size': 14, 'axes.titlesize': 16, 'axes.labelsize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
}
mpl.rcParams.update(POSTER_RC)
POSTER_DIR = '/home/yacheng/ssl_outthere/poster_figs'
os.makedirs(POSTER_DIR, exist_ok=True)

# robust scatter of the residuals, in the target's own units (dex for log targets).
# Same estimator as LowResPT's sigma_NMAD, without the redshift-only (1+z) normalization.
def sigma_nmad(t, p):
    d = p - t
    return 1.4826 * np.median(np.abs(d - np.median(d)))

PANEL_C = ['#006bff', 'darkorange', 'seagreen']       # one colour per Sérsic target
PAD = 0.05                                            # extra axis margin (fraction of range)

fig, axes = plt.subplots(1, len(TARGETS), figsize=(5 * len(TARGETS), 4.8))
for j, (ax, (col, label, _)) in enumerate(zip(axes, TARGETS)):
    t, p = Yte[:, j], preds[:, j]
    c = PANEL_C[j % len(PANEL_C)]
    lo, hi = np.percentile(np.concatenate([t, p]), [0.5, 99.5])
    m = PAD * (hi - lo); lo, hi = lo - m, hi + m
    ax.scatter(t, p, s=6, alpha=0.2, edgecolors='none', color=c)
    ax.plot([lo, hi], [lo, hi], '--', lw=2, color='black', alpha=0.4, label='1:1')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel(rf'true  {label}'); ax.set_ylabel(rf'predicted  {label}')
    r2 = r2_score(t, p); snmad = sigma_nmad(t, p)
    ax.text(0.97, 0.03,
            rf'$R^2$={r2:.3f}' + '\n' + rf'$\sigma_{{\rm NMAD}}$={snmad:.4f}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=13, color=c,
            fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.7, pad=3.0, edgecolor='none'))
    if j == 0:
        ax.legend(fontsize=12, loc='upper left')
    for sp in ax.spines.values():
        sp.set_linewidth(1.8)
    ax.tick_params(width=1.6)
fig.patch.set_alpha(0.0)
fig.tight_layout()
outp = os.path.join(POSTER_DIR, 'sersic_linear_probe')
fig.savefig(outp + '.png', transparent=False, dpi=300, bbox_inches='tight')
print(f'Saved -> {outp}.png')
plt.show()

## Zero-shot k-NN probe

Non-parametric readout on the *same* frozen embeddings and the *same* train/test
split as the linear probe: a test object's label is the mean label of its $k$
nearest training neighbours in embedding space. Nothing is fitted, so this asks
whether the space already organises galaxies by the Sérsic parameters, rather
than whether a linear map can be found. Neighbour search runs on the GPU in
chunks; both cosine and Euclidean metrics are swept over a range of $k$.


In [ ]:
# zero-shot k-NN regression on the frozen embeddings (same split as the linear probe)
KS      = [1, 3, 5, 10, 20, 50, 100]
METRICS = ['cosine', 'euclidean']
CHUNK   = 2048                                  # test rows per distance block

def knn_predict(Ztr, Zte, Ytr, ks, metric, device=DEVICE):
    """Mean-of-neighbours prediction for every k in ks; returns {k: (n_te, n_targets)}."""
    A = torch.as_tensor(Ztr, dtype=torch.float32, device=device)
    B = torch.as_tensor(Zte, dtype=torch.float32, device=device)
    L = torch.as_tensor(Ytr, dtype=torch.float32, device=device)
    if metric == 'cosine':
        A = torch.nn.functional.normalize(A, dim=1)
        B = torch.nn.functional.normalize(B, dim=1)
    else:
        a2 = (A * A).sum(1)                     # ||a||^2, the only train-side term needed
    kmax = max(ks)
    out = {k: [] for k in ks}
    for s in range(0, B.shape[0], CHUNK):
        b = B[s:s + CHUNK]
        if metric == 'cosine':
            score = b @ A.T                     # larger is closer
        else:
            score = -(a2[None, :] - 2.0 * (b @ A.T))   # -||a-b||^2 up to a per-row constant
        idx = score.topk(kmax, dim=1).indices
        nb  = L[idx]                            # (chunk, kmax, n_targets)
        for k in ks:
            out[k].append(nb[:, :k].mean(1).cpu().numpy())
    return {k: np.concatenate(v) for k, v in out.items()}

results = {}                                    # (metric, k) -> per-target (R2, sigma_NMAD)
for metric in METRICS:
    P = knn_predict(Ztr, Zte, Ytr, KS, metric)
    hdr = f'{"k":>4s} ' + ' '.join(f'{c:>22s}' for c, _, _ in TARGETS)
    print(f'=== zero-shot k-NN [{metric}]  ({len(Xtr)} reference / {len(Xte)} test) ===')
    print(hdr); print('-' * len(hdr))
    for k in KS:
        row = []
        for j, (col, _, _) in enumerate(TARGETS):
            r2 = r2_score(Yte[:, j], P[k][:, j])
            sn = sigma_nmad(Yte[:, j], P[k][:, j])
            results[(metric, k, col)] = (r2, sn)
            row.append(f'R2={r2:6.3f} nmad={sn:5.3f}')
        print(f'{k:>4d} ' + ' '.join(f'{r:>22s}' for r in row))
    print()

# R^2 vs k, one line per target and per metric
fig, axes = plt.subplots(1, len(TARGETS), figsize=(4.6 * len(TARGETS), 3.8), sharex=True)
for j, (ax, (col, label, _)) in enumerate(zip(axes, TARGETS)):
    for metric, ls in zip(METRICS, ['-', '--']):
        ax.plot(KS, [results[(metric, k, col)][0] for k in KS], ls, marker='o', ms=4,
                color=PANEL_C[j % len(PANEL_C)], alpha=1.0 if ls == '-' else 0.5,
                label=f'{metric} k-NN')
    lin_r2 = r2_score(Yte[:, j], preds[:, j])
    ax.axhline(lin_r2, color='0.35', lw=1.2, ls=':', label='linear probe')
    ax.set_xscale('log'); ax.set_xlabel('k'); ax.set_title(label)
    if j == 0:
        ax.set_ylabel(r'$R^2$'); ax.legend(fontsize=10, loc='lower left')
    for sp in ax.spines.values():
        sp.set_linewidth(1.8)
    ax.tick_params(which='both', direction='in', width=1.6, top=True, right=True)
fig.patch.set_alpha(0.0)
fig.tight_layout()
outp = os.path.join(POSTER_DIR, 'sersic_knn_vs_k')
fig.savefig(outp + '.png', transparent=False, dpi=300, bbox_inches='tight')
print(f'Saved -> {outp}.png')
plt.show()


### Zero-shot pred-vs-true at a fixed k

Same panel layout as the linear-probe figure, at a single moderate `KNN_K`
(the $R^2(k)$ curves plateau from $k\simeq10$; $k=5$ is kept to match the 5-NN
probe used for the spectrum encoder). The linear-probe $R^2$ is quoted in each
panel for reference.


In [ ]:
# zero-shot k-NN pred-vs-true at a single k, styled like the linear-probe figure
KNN_K      = 5              # k used for the figure; the R2(k) scan above plateaus at ~10-20
KNN_METRIC = 'euclidean'    # cosine and euclidean agree to <0.01 in R2

P_k = knn_predict(Ztr, Zte, Ytr, [KNN_K], KNN_METRIC)[KNN_K]

fig, axes = plt.subplots(1, len(TARGETS), figsize=(5 * len(TARGETS), 4.8))
for j, (ax, (col, label, _)) in enumerate(zip(axes, TARGETS)):
    t, p = Yte[:, j], P_k[:, j]
    c = PANEL_C[j % len(PANEL_C)]
    lo, hi = np.percentile(np.concatenate([t, p]), [0.5, 99.5])
    m = PAD * (hi - lo); lo, hi = lo - m, hi + m
    ax.scatter(t, p, s=6, alpha=0.2, edgecolors='none', color=c)
    ax.plot([lo, hi], [lo, hi], '--', lw=2, color='black', alpha=0.4, label='1:1')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel(rf'true  {label}'); ax.set_ylabel(rf'predicted  {label}')
    r2, snmad = r2_score(t, p), sigma_nmad(t, p)
    ax.text(0.97, 0.03,
            rf'$R^2$={r2:.3f}' + '\n' + rf'$\sigma_{{\rm NMAD}}$={snmad:.4f}' + '\n'
            + rf'(linear: $R^2$={r2_score(t, preds[:, j]):.3f})',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=13, color=c,
            fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.7, pad=3.0, edgecolor='none'))
    ax.set_title(f'{KNN_K}-NN zero-shot' if j == 1 else '')
    if j == 0:
        ax.legend(fontsize=12, loc='upper left')
    for sp in ax.spines.values():
        sp.set_linewidth(1.8)
    ax.tick_params(width=1.6)
fig.patch.set_alpha(0.0)
fig.tight_layout()
outp = os.path.join(POSTER_DIR, f'sersic_knn{KNN_K}_zeroshot')
fig.savefig(outp + '.png', transparent=False, dpi=300, bbox_inches='tight')
print(f'Saved -> {outp}.png')
plt.show()
